# 第 23 章 · 媒体：附件身份、预算与产物提交

图片可以已经显示，但尚未送给模型；服务可以已经返回图片，但尚未保存为项目产物。用一枚 success 标志覆盖所有阶段，会掩盖失效点。

本章用 Pillow 生成纯色合成输入，真实解码、缩放、哈希、落盘及恢复。合成 PNG 是已知字节，**没有调用图像生成服务或视觉模型**，也不用于证明图像质量。

**运行范围：局部机制实验。** 按顺序执行，每个实验都通过真实函数、文件或队列产生结果，再用断言检查。修改参数后重跑该实验，必要时先重启内核并执行全部。所有合成输入都只证明本地机制；本章没有累计检查点，真实服务与设备仍待验证。

本章使用原作者的设计与局部原型；下面明确写出的简化是教材选择，不是 Codex 原样实现。固定源码：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter

## 1 · 内容地址与可变路径分开

同一字节应得到同一附件 ID；图片宽高从解码得出，不信任文件扩展名。改变 `width`、`height` 或颜色重跑，观察内容身份变化。预览回执明确保留“尚未提交模型”。

In [2]:
from io import BytesIO
from PIL import Image
m = load_example('23', 'media_core')
run_dir = fresh_run_dir('ch23')
def png(width=640, height=320, color=(30, 120, 200, 255)):
    stream = BytesIO()
    Image.new('RGBA', (width, height), color).save(stream, format='PNG')
    return stream.getvalue()
store = m.AttachmentStore(run_dir/'attachments')
original = store.ingest(png(), 'image/png', 'synthetic-input')
duplicate = store.ingest(png(), 'image/png', 'second-path')
assert original['id'] == duplicate['id']
assert (original['width'], original['height']) == (640, 320)
preview = store.preview(original)
assert preview['model_delivery'] == 'not_submitted'
print({'id': original['id'], 'size': original['size'], 'preview': preview})

{'id': 'sha256:53a3d2ddff07769f5806b44e351ba1ea602fc0b51092ebb09113ec2dacdd4708', 'size': 1571, 'preview': {'attachment_id': 'sha256:53a3d2ddff07769f5806b44e351ba1ea602fc0b51092ebb09113ec2dacdd4708', 'preview_bytes': 1571, 'model_delivery': 'not_submitted'}}


## 2 · 能力检查在编码之前，预算检查在多个层次

像素、文件字节和 base64 后的传输字节不是同一个量。缩放保留父附件 ID，便于追溯。修改 `max_dimension` 到 160，观察 prepared 的宽高；设置极小 `max_wire_bytes` 则应明确拒绝。

In [3]:
try:
    store.prepare_content(original, frozenset({'text'}))
except m.MediaError as error:
    assert str(error) == 'unsupported_model_modality'
else:
    raise AssertionError('不支持图像的模型不能接收图片')
prepared, content = store.prepare_content(original, frozenset({'image'}), max_dimension=320)
assert (prepared['width'], prepared['height']) == (320, 160)
assert prepared['parent'] == original['id']
assert content['type'] == 'input_image'
try:
    store.prepare_content(original, frozenset({'image'}), max_wire_bytes=32)
except m.MediaError as error:
    assert str(error) == 'wire_budget'
else:
    raise AssertionError('传输预算未生效')
print({'dimensions': (prepared['width'], prepared['height']), 'wire_bytes': len(content['image_url'])})

{'dimensions': (320, 160), 'wire_bytes': 838}


## 3 · provider 完成与本地产物提交各有状态

`complete` 接受本地合成的服务回包字节，只验证接收逻辑。先故意占用目标路径，确认本地保存失败不会把远端成功抹掉；再换路径保存，并从磁盘恢复同一个 job。

In [4]:
jobs = m.Jobs(run_dir/'jobs', store)
job = m.ImageJob('job1', 'edit', [original], True)
request = jobs.request(job, '合成输入的协议实验')
assert request['images'][0]['sha256'] == original['sha256']
jobs.complete(job, png(color=(60, 180, 120, 128)), 'synthetic-provider-1')
assert job.remote == 'succeeded' and job.delivery == 'staged'
destination = run_dir/'artifacts'/'existing.png'
destination.parent.mkdir()
destination.write_bytes(b'keep existing file')
jobs.save_artifact(job, destination)
assert job.remote == 'succeeded' and job.delivery == 'failed'
assert destination.read_bytes() == b'keep existing file'
jobs.save_artifact(job, run_dir/'artifacts'/'result.png')
assert job.delivery == 'committed'
restored = jobs.restore('job1')
assert restored.artifact == job.artifact
reference = m.attach_readme(restored, run_dir)
assert (run_dir/reference['target']).is_file()
print({'remote': restored.remote, 'delivery': restored.delivery, 'reference': reference})

{'remote': 'succeeded', 'delivery': 'committed', 'reference': {'readme': '<project>/runs/notebooks/ch23-20261113T124123Z-4ef778d7/README.md', 'target': 'artifacts/result.png', 'sha256': '7912423546913e3d48ac175c4cd00b45ac20d755d291e5f2a6c98f18db89f18f'}}


## 4 · 取消后的晚到成功不能写项目

本地取消不能宣称远端已终止。结果随后到达可以留在附件存储供核对，但不再提交项目路径。恢复后沿用请求身份，拒绝重新发起一次可能重复收费的生成。

In [5]:
late = m.ImageJob('late1', 'generate', [], False)
jobs.request(late, '受控晚到结果')
jobs.cancel(late)
assert late.remote == 'unknown'
jobs.complete(late, png(80, 80), 'synthetic-late')
jobs.save_artifact(late, run_dir/'must-not-exist.png')
assert late.remote == 'succeeded' and late.delivery == 'cancelled_before_commit'
assert not (run_dir/'must-not-exist.png').exists()
restored_late = jobs.restore('late1')
try:
    jobs.request(restored_late, '危险的自动重试')
except m.MediaError as error:
    assert str(error) == 'unsafe_resubmit'
else:
    raise AssertionError('恢复不应重新生成')
bad = m.ImageJob('bad1', 'generate', [], False)
jobs.request(bad, '无效 payload')
jobs.complete(bad, b'not an image', 'synthetic-bad')
assert bad.remote == 'succeeded' and bad.delivery == 'invalid_or_unstaged_result'
print({'late': restored_late.delivery, 'invalid_payload': bad.delivery})

{'late': 'cancelled_before_commit', 'invalid_payload': 'invalid_or_unstaged_result'}


## 源码对照与接回主线

以下为固定提交的已定位源码入口。阅读源码能解释设计边界，不等于执行过其测试；本章 Python 原型与这些 Rust 接口不是同一实现。

- [codex-rs/protocol/src/models.rs#L950-L1013](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/protocol/src/models.rs#L950-L1013)
- [codex-rs/attachment-store/src/lib.rs#L28-L114](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/attachment-store/src/lib.rs#L28-L114)
- [codex-rs/core/src/tools/handlers/view_image.rs#L94-L206](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/tools/handlers/view_image.rs#L94-L206)

**接回条件：** 将局部机制接入同一 Dispatcher、Scheduler、History 和受保护验证器，再执行原合同。目录、断言和演示输出都不能替代累计集成证据。